In [ ]:
##this script is used for extracting the ms2 data from the targetd msms data and used for spectrum comparison.

In [10]:
#helper functions
# !pip install pyopenms
# !pip install numpy
# !pip install tabulate
# !pip install matchms
import re
import numpy as np
import matchms 
from matchms import calculate_scores
from matchms import Spectrum
from matchms.similarity import CosineGreedy
import pandas as pd
import pyopenms as oms
import tabulate
import os 
import copy
import warnings
import logging
# Suppress specific warnings
warnings.filterwarnings("ignore", message=".*")

def ppm_error(mz, target):
    return abs(mz - target) / target * 1e6

def preprocess_spectra(filepath, target_mz, ce_level, tolerance=5):
    # Extract spectra with specified collision energy level for target m/z 
    if target_mz is None:
        raise ValueError("Please provide a target m/z value.")
    ce_level = float(ce_level)
    tolerance = float(tolerance)
    target_mz = float(target_mz)
    spectra = oms.MSExperiment()
    oms.MzMLFile().load(filepath, spectra)
    ms2_spec = spectra.getSpectra()

    ms2_query = oms.MSExperiment()
    for s in ms2_spec:
        if s.getMSLevel() == 1:
            continue
        precursors = s.getPrecursors()
        mslevel = s.getMSLevel()
        collision_energy = precursors[0].getMetaValue("collision energy")
        ms_error = ppm_error(precursors[0].getMZ(), target_mz)
        if ms_error < tolerance and mslevel == 2 and collision_energy == ce_level:
            ms2_query.addSpectrum(s)
        else :
            continue

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]

    # print(f'Number of MS2 spectra before merge: {len(ms2beforemerg)}')

    if len(ms2beforemerg) == 0:
        # print(f"No spectra found with specified parameters for {ce_level}.")
        return None
    
    elif len(ms2beforemerg) > 1:
        #merget the ms spectra
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("mz_tolerance", 1e-3)
        param.setValue("rt_tolerance", "5.0")
        merger.setParameters(param)
        merger.mergeSpectraPrecursors(ms2_query)
        return ms2_query
    elif len(ms2beforemerg) == 1:
        return ms2_query
    


def parse_msp_file(msp_file_path, polarity):
    """
    Parses the .msp file to extract all spectrum information.
    
    Args:
        msp_file_path (str): Path to the .msp file.

    Returns:
        dict: A dictionary where keys are InChIKey or SMILES and values are lists of spectra data.
    """
    spectra = {}
    if polarity == "positive":
        p_type = "[M+H]+"
    elif polarity == "negative":
        p_type = "[M-H]-"
    else:
        raise ValueError("Invalid polarity. Please specify 'positive' or 'negative'.")
    
    with open(msp_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('Name: ')
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata = {}
            spectrum_data = []
            key = None

            for line in lines:
                if line.startswith("InChIKey:"):
                    key = line.split(": ")[1].strip()
                elif line.startswith("SMILES:") and key is None:
                    key = line.split(": ")[1].strip()
                elif line.startswith("Spectrum_type:"):
                    spectrum_type = line.split(": ")[1].strip()
                elif line.startswith("Precursor_type:"):
                    precursor_type = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    spectrum_data = lines[lines.index(line)+1:lines.index(line)+1+num_peaks]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            # Retain only spectra with "Spectrum_type: MS2"
            if spectrum_type == "MS2" and precursor_type == p_type  and key and spectrum_data:
                if key not in spectra:
                    spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": [(float(mz), float(intensity)) for mz, intensity in (line.split() for line in spectrum_data)]
                })

    return spectra


def parse_mona_database(db_file_path, polarity):
    """
    Parses an alternate database format to extract all spectrum information based on polarity.

    Args:
        db_file_path (str): Path to the database file.
        polarity (str): The ion polarity to filter ("positive" or "negative").

    Returns:
        dict: A dictionary where keys are InChIKey or SMILES and values are lists of spectra data.
    """
    spectra = {}

    if polarity == "positive":
        p_type = "[M+H]+"
    elif polarity == "negative":
        p_type = "[M-H]-"
    else:
        raise ValueError("Invalid polarity. Please specify 'positive' or 'negative'.")

    with open(db_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('NAME: ')  # "NAME:" marks the beginning of each spectrum.
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata = {}
            spectrum_data = []
            key = None

            for line in lines:
                if line.startswith("INCHIKEY:"):
                    key = line.split(": ")[1].strip()
                elif line.startswith("SMILES:") and key is None:
                    key = line.split(": ")[1].strip()
                elif line.startswith("PRECURSORTYPE:"):
                    ionization = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    spectrum_data = [tuple(map(float, peak.split())) for peak in lines[lines.index(line)+1:] if peak.strip()]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            # Retain only spectra matching the desired polarity
            if ionization == p_type and key and spectrum_data:
                if key not in spectra:
                    spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": spectrum_data
                })

    return spectra


def get_spectrum_by_key(spectra, key):
    """
    Retrieves all spectrum information by InChIKey or SMILES.

    Args:
        spectra (dict): Parsed spectra data.
        key (str): The InChIKey or SMILES to search for.

    Returns:
        dict: A dictionary where keys are collision energies and values are spectra.
    """
    if key not in spectra:
        return {}

    grouped_spectra = {}
    for spec in spectra[key]:
        collision_energy = spec["metadata"].get("Collision_energy", spec["metadata"].get("COLLISIONENERGY", "Unknown"))
        if collision_energy not in grouped_spectra:
            grouped_spectra[collision_energy] = []
        grouped_spectra[collision_energy].append(spec["spectrum"])

    return grouped_spectra

def normalize_and_filter(spectrum, baseline):
    """
    Accepts:
      - pyopenms/matchms Spectrum with .get_peaks()
      - list of (mz, intensity) tuples
      - tuple: (mz_array, intensity_array)
    Returns:
      - (mz_res, int_res): top-N peaks (sorted by m/z)
    """
    # -------- 1. unify input format: get mz and intensity as np.arrays --------
    if hasattr(spectrum, "get_peaks"):
        # pyOpenMS or matchms Spectrum
        mz, intensity = spectrum.get_peaks()
        mz = np.asarray(mz, dtype=float)
        intensity = np.asarray(intensity, dtype=float)

    elif isinstance(spectrum, list) and len(spectrum) > 0 and isinstance(spectrum[0], (tuple, list)):
        # list of (mz, intensity) pairs
        mz, intensity = zip(*spectrum)
        mz = np.asarray(mz, dtype=float)
        intensity = np.asarray(intensity, dtype=float)

    elif isinstance(spectrum, tuple) and len(spectrum) == 2:
        # (mz_array, intensity_array)
        mz = np.asarray(spectrum[0], dtype=float)
        intensity = np.asarray(spectrum[1], dtype=float)

    else:
        raise ValueError("Spectrum format is not recognized. "
                         "Expected object with .get_peaks(), list of (mz,intensity), or (mz_array,int_array) tuple.")

    # -------- 2. select top 8 most intense peaks --------
    if len(intensity) > 8:
        top_indices = np.argsort(intensity)[-8:]
    else:
        top_indices = np.argsort(intensity)

    mz_res = mz[top_indices]
    int_res = intensity[top_indices]

    # -------- 3. sanity check & sort by m/z --------
    if len(mz_res) != len(int_res):
        raise ValueError("The length of mz and intensity arrays do not match after filtering.")

    sorted_indices = np.argsort(mz_res)
    mz_res = mz_res[sorted_indices]
    int_res = int_res[sorted_indices]

    return mz_res, int_res

# def normalize_and_filter(spectrum, baseline):
#     try:
#         mz, intensity = spectrum.get_peaks()
#     except AttributeError: 
#         if isinstance(spectrum, list) and all(isinstance(i, tuple) for i in spectrum):
#             mz, intensity = zip(*spectrum)
#             mz = np.array(mz)
#             intensity = np.array(intensity)
#         else:
#             raise ValueError("Spectrum format is not recognized.")
#     except:
#         if isinstance(spectrum, tuple) and len(spectrum) == 2:
#             mz = np.array(spectrum[0])
#             intensity = np.array(spectrum[1])
#         else:
#             raise ValueError("Spectrum format is not recognized.")
    
#     # select the top 8 most intense peaks
#     if len(intensity) > 8:
#         top_indices = np.argsort(intensity)[-8:]
#         mz_res = mz[top_indices]
#         int_res = intensity[top_indices]
#     else:
#         top_indices = np.argsort(intensity)
#         mz_res = mz[top_indices]
#         int_res = intensity[top_indices]

#     # Assert that the length of mz and intensity are the same
#     if len(mz_res) != len(int_res):
#         raise ValueError("The length of mz and intensity arrays do not match after filtering.")
    
#     # Sort mz values in ascending order and reorder intensities correspondingly.
#     sorted_indices = np.argsort(mz_res)
#     mz_res = mz_res[sorted_indices]
#     int_res = int_res[sorted_indices]
#     return mz_res, int_res

def find_best_match(references, queries, tolerance, key):
    """
    Calculate cosine‐greedy scores and return the best match info,
    or None for everything if no references or on error.
    """
    if not references:
        return None, None, None, None

    try:
        scores = matchms.calculate_scores(
            references=references,
            queries=queries,
            similarity_function=CosineGreedy(tolerance=tolerance),
            is_symmetric=False
        )

        best_scores, num_match_peaks, best_refs, best_queries = [], [], [], []
        for query in queries:
            matches = scores.scores_by_query(query, 'CosineGreedy_score', sort=True)
            for ref, (score, n_peaks) in matches:
                best_scores.append(score)
                num_match_peaks.append(n_peaks)
                best_refs.append(ref.metadata['peak_comments'])
                best_queries.append(query.metadata['peak_comments'])

        if not best_scores:
            return None, None, None, None
        
        #filter mathces with num_match_peaks <2 and find the maximum score among the rest
        filtered_scores = [(s, n, r, q) for s, n, r, q in zip(best_scores, num_match_peaks, best_refs, best_queries) if n >=2]
        if not filtered_scores:
            return None, None, None, None
        best_scores, num_match_peaks, best_refs, best_queries = zip(*filtered_scores)
        idx = best_scores.index(max(best_scores))
        return (
            best_scores[idx],
            num_match_peaks[idx],
            best_refs[idx],
            best_queries[idx],
        )

    except (IndexError, ValueError):
        # print(f'No match found for compound {key}')
        return None, None, None, None

In [1]:
!pip install -e "D:\NTA_analysis\LCMS_data_processing_utils[dev]"

Obtaining file:///D:/NTA_analysis/LCMS_data_processing_utils
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lcms-data-processing-utils (pyproject.toml): started
  Building editable for lcms-data-processing-utils (pyproject.toml): finished with status 'done'
  Created wheel for lcms-data-processing-utils: filename=lcms_data_processing_utils-0.1.0-0.editable-py3-none-any.whl size=4159 sha256=1dd5637938f95d98d02a771fc5ede6b9853b63633a112ebf1cc09b862ac045eb
  Stored in directory: C:\Users\yangj\AppData\Local\Temp\pip-ephem-wh


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
##restart the kernel and import my_package
import my_package

In [13]:
#load database from pickle file
from my_package.spectrum_utils import parse_mona_database

import pickle
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mbank_data_neg.pkl', 'rb') as f:
    mbank_data_neg = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_data_neg.pkl', 'rb') as f:
    mona_data_neg = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_obtrap_data_neg.pkl', 'rb') as f:
    mona_obtrap_data_neg = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mbank_data_pos.pkl', 'rb') as f:
    mbank_data_pos = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_data_pos.pkl', 'rb') as f:
    mona_data_pos = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_obtrap_data_pos.pkl', 'rb') as f:
    mona_obtrap_data_pos = pickle.load(f)

#loading inhouse library
inhouse_library_neg= r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\ENTACT_mix_standards\ENTACT_neg_spectra.msp"
inhouse_library_pos= r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\ENTACT_mix_standards\ENTACT_pos_spectra.msp"
inhouse_lib2 = r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\new_mixture_spectra.msp"

inhouse_data_neg = parse_mona_database(inhouse_library_neg, polarity="negative")
inhouse_data_pos = parse_mona_database(inhouse_library_pos, polarity="positive")
inhouse_data_neg2 = parse_mona_database(inhouse_lib2, polarity="negative")
inhouse_data_pos2 = parse_mona_database(inhouse_lib2, polarity="positive")

In [ ]:
#import mzml files for spectrum matching
manualcheck_neg = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\NegPool_11172025\neg_msms_method_manualcheck.csv'
manulacheck_pos = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\PosPool_12012025\pos_msms_method_manualcheck.csv'
neg_check = pd.read_csv(manualcheck_neg)
pos_check = pd.read_csv(manulacheck_pos)
matches_neg = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv')
matches_pos = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv')

import tabulate
#mutate feature_id column by combining average rt and average mass
matches_neg['feature_id'] = matches_neg['Average Rt(min)'].astype(str) + "_" + matches_neg['Average Mz'].astype(str)
matches_pos['feature_id'] = matches_pos['Average Rt(min)'].astype(str) + "_" + matches_pos['Average Mz'].astype(str)
firstmatch_neg = matches_neg[matches_neg['feature_id'].isin(neg_check['feature_id'])]
firstmatch_pos = matches_pos[matches_pos['feature_id'].isin(pos_check['feature_id'])]
#concate the two dataframes, to acquire msfile path, RT, and inchikey for potential candidates
firstmatch_neg = pd.merge(firstmatch_neg, neg_check[['feature_id', 'MS2_filepath']], on='feature_id', how='left')
firstmatch_pos = pd.merge(firstmatch_pos, pos_check[['feature_id', 'MS2_filepath']], on='feature_id', how='left')


+----+--------------+-------------------+--------------+-----------------+---------------+-----------------------------+-------------------------------------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+-----------------------------+-----------------------------+-------------------+----------------+---------------------+-------------+---------------+----------+----------+------------------+------------------------+------------------+-------------------------------+------------------------+-------------------------+----------------------------------------------------------------------------------------------------------------------+
|    |   Unnamed: 0 |   Average Rt(min) |   Average Mz | feature_id      | DTXSID_x      | SMILES_STD                  | PREFERRED_NAME_x                                | MOLECULAR_FORMULA_original   | Pred. Ionization source   |   BloodPaperCount | 2019 PV   | InChiKey_o

In [ ]:
print(matches_neg.shape)
print(matches_pos.shape)
print(matches_neg['priority_confirmation'].value_counts())
print(matches_pos['priority_confirmation'].value_counts())

(27355, 27)
(36300, 27)
priority_confirmation
4    20070
3     7052
1      233
Name: count, dtype: int64
priority_confirmation
4    23769
3     7608
2     4651
1      272
Name: count, dtype: int64


In [ ]:
#import filtered chemical list
massspec_neg = pd.read_csv("D:/UCSF_postdoc_topic/REVEAL_first_200/chemical_list/toxtarget_with_spectrum_neg.csv")
massspec_pos = pd.read_csv("D:/UCSF_postdoc_topic/REVEAL_first_200/chemical_list/toxtarget_with_spectrum_pos.csv")

#fitler firstbatch_neg with massspec_neg by INCHIKEY
firstbatch_neg = firstmatch_neg[firstmatch_neg['InChiKey_origin'].isin(massspec_neg['INCHIKEY'])]  
print(f"Number of features in firstbatch_neg after filtering with massspec_neg: {len(firstbatch_neg)}")

firstbatch_pos = firstmatch_pos[firstmatch_pos['InChiKey_origin'].isin(massspec_pos['INCHIKEY'])]
print(f'number of chemical in firstmatch_pos after filtering with masspec_pos:{len(firstbatch_pos)}')

Number of features in firstbatch_neg after filtering with massspec_neg: 20
number of chemical in firstmatch_pos after filtering with masspec_pos:153


In [ ]:
#clean up the script and make filter on the number of matched peaks
##set threshold for peak matching
match_tolerance = 0.005 #dalton
peak_int_tol = 5 #ppm
peak_norm_tol = 5 #%
##import library for spectrum comparison
#perform spectrum search for each row by InChiKey_origin column, and add number of found spectra to the dataframe from each database.
#add tqdm progress bar to the loop

# suppress all matchms warnings
logging.getLogger("matchms").setLevel(logging.ERROR)
##set threshold for peak matching
match_tolerance = 0.01 #dalton
precursor_mass_tol = 10 #ppm
peak_norm_tol = 5 #%
##import library for spectrum comparison
#perform spectrum search for each row by InChiKey_origin column, and add number of found spectra to the dataframe from each database.
#add tqdm progress bar to the loop`

# suppress all matchms warnings
logging.getLogger("matchms").setLevel(logging.ERROR)

for iter, row in firstmatch_neg.iterrows():
    key = row['InChiKey_origin']

    #extract the query spectrum from targeted injection
    query_spectrums =[]
    target_mz = row['Average Mz']
    q_filepath = row['MS2_filepath']

    celevel = [10,20,40]
    for ce in celevel:
        ms2_query = preprocess_spectra(q_filepath, target_mz, ce, tolerance=precursor_mass_tol)
        if ms2_query is None:
            # print(f'No spectra found for compound {key} with CE {ce}')
            continue
        sspectrum = Spectrum(mz = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[0],
                             intensities = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[1], metadata = {"inchikey": key, 'peak_comments':  str(ce)+'eV'})
        query_spectrums.append(sspectrum)

    if len(query_spectrums) ==0:
        print(f'No query spectrum found for compound {key}')
        continue

    #extract the reference spectrum from database
    reference_spectrums_library = []
    reference_spectrums_insilico =[]

    sources = [
    (mbank_data_neg, reference_spectrums_library,  'mbank'),
    (mona_data_neg,  reference_spectrums_library,   'mona'),
    (mona_obtrap_data_neg, reference_spectrums_library, 'mona_obtrap'),
    (inhouse_data_neg, reference_spectrums_library, 'inhouse1'),
    (inhouse_data_neg2, reference_spectrums_library, 'inhouse2')]


    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
    reference_spectrums_library,
    query_spectrums,
    match_tolerance,
    key)
    
    firstmatch_neg.loc[iter, 'library_best_match'] = best_spectrum_library
    firstmatch_neg.loc[iter, 'library_best_match_num_matchpeak'] = best_num_matchpeak_library
    firstmatch_neg.loc[iter, 'library_best_match_query'] = best_match_query_library
    firstmatch_neg.loc[iter, 'library_best_match_score_value'] = best_score_library

In [10]:
firstmatch_neg['library_best_match_score_value'].fillna(0, inplace=True)
# print(tabulate.tabulate(firstmatch_neg.head(10), headers='keys', tablefmt='psql'))

#get the summary of the library_best_match_score_value column
summary_neg = firstmatch_neg['library_best_match_score_value'].describe()
print("Summary of library_best_match_score_value:")
print(summary_neg)

Summary of library_best_match_score_value:
count    32.000000
mean      0.463953
std       0.450291
min       0.000000
25%       0.000000
50%       0.462995
75%       0.933755
max       0.999758
Name: library_best_match_score_value, dtype: float64


In [88]:
#filter rows with library_best_match_num_matchpeak >= 3, 
firstmatch_neg_filter = firstmatch_neg[firstmatch_neg['library_best_match_num_matchpeak']>=3]
print(tabulate.tabulate(firstmatch_neg_filter, headers='key', tablefmt= 'psql'))

+----+--------+-------+---------+-----------------+---------------+-----------------------------+-----------------------+----------+-----+------+-----+-----------------------------+-----------------------------+-----------------------------+-------+---------+-----+-------------+------+---------+---------+-----------------+---------+------------------+------------------+------------------+----+----------------------------------------------------------------------------------------------------------------------+--------------------------+----+------+----------+-----------------------------+
|    |        |       |         |                 |               |                             |                       |          |     |      |     |                             |                             |                             |       |         |     |             |      |         |         |                 |         |                  |                  |                  |    |        

In [ ]:
# suppress all matchms warnings
logging.getLogger("matchms").setLevel(logging.ERROR)

for iter, row in firstmatch_pos.iterrows():
    key = row['InChiKey_origin']

    #extract the query spectrum from targeted injection
    query_spectrums =[]
    target_mz = row['Average Mz']
    q_filepath = row['MS2_filepath']

    celevel = [10,20,40]
    for ce in celevel:
        ms2_query = preprocess_spectra(q_filepath, target_mz, ce, tolerance=precursor_mass_tol)
        if ms2_query is None:
            # print(f'No spectra found for compound {key} with CE {ce}')
            continue
        sspectrum = Spectrum(mz = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[0],
                             intensities = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[1], metadata = {"inchikey": key, 'peak_comments':  str(ce)+'eV'})
        query_spectrums.append(sspectrum)

    if len(query_spectrums) ==0:
        print(f'No query spectrum found for compound {key}')
        continue

    #extract the reference spectrum from database
    reference_spectrums_library = []
    reference_spectrums_insilico =[]

    sources = [
    (mbank_data_pos, reference_spectrums_library,  'mbank'),
    (mona_data_pos,  reference_spectrums_library,   'mona'),
    (mona_obtrap_data_pos, reference_spectrums_library, 'mona_obtrap'),
    (inhouse_data_pos, reference_spectrums_library, 'inhouse1'),
    (inhouse_data_pos2, reference_spectrums_library, 'inhouse2')]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
    reference_spectrums_library,
    query_spectrums,
    match_tolerance,
    key)

    # best_score_insilico, best_num_matchpeak_insilico, best_spectrum_insilico, best_match_query_insilico = find_best_match(
    #     reference_spectrums_insilico,
    #     query_spectrums,
    #     match_tolerance,
    #     key)
    
    firstmatch_pos.loc[iter, 'library_best_match'] = best_spectrum_library
    firstmatch_pos.loc[iter, 'library_best_match_num_matchpeak'] = best_num_matchpeak_library
    firstmatch_pos.loc[iter, 'library_best_match_query'] = best_match_query_library
    firstmatch_pos.loc[iter, 'library_best_match_score_value'] = best_score_library

In [ ]:
#output matching results from the first match with targeted msms
postarget_copy = firstmatch_pos.copy()
negtarget_copy = firstmatch_neg.copy()

postarget_copy['matching_by'] = 'targetedmsms'
negtarget_copy['matching_by'] = 'targetedmsms'
# postarget_copy.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\positive_matches_targetedmsms_from12012025.csv')
# negtarget_copy.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\negative_matches_targetedmsms_from11172025.csv')

#identification from targeted msms


In [15]:
#compare RT and Spectrum fron inhouse library, and draw spectrum and EIC graph
postarget_copy = firstmatch_pos.copy()
negtarget_copy = firstmatch_neg.copy()

# filepath = r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\inhouse_RTMS_library_Nov.csv"
# RTMS_library = pd.read_csv(filepath)

# postarget_copy = postarget_copy.merge(RTMS_library[['INCHIKEY', 'RT_pos']], on= 'INCHIKEY', how='left')
# negtarget_copy = negtarget_copy.merge(RTMS_library[['INCHIKEY','RT_neg']], on = 'INCHIKEY', how = 'left')

postarget_copy_filter = postarget_copy[(postarget_copy['library_best_match_num_matchpeak'] >= 3.0)&(postarget_copy['library_best_match_score_value']>=0.7)]
print(postarget_copy_filter.shape)
postarget_copy_filter['RT_diff'] = abs(postarget_copy_filter['Average Rt(min)'] - postarget_copy_filter['RT_pos'])
print(postarget_copy_filter[postarget_copy_filter['RT_diff'] <=0.5].shape)
print(tabulate.tabulate(postarget_copy_filter[postarget_copy_filter['RT_diff'] <=0.5], headers='keys', tablefmt='psql'))
print(tabulate.tabulate(postarget_copy_filter[postarget_copy_filter['RT_diff'].isna()], headers='keys', tablefmt='psql'))

negtarget_copy_filter = negtarget_copy[(negtarget_copy['library_best_match_num_matchpeak'] >= 3.0) & (negtarget_copy['library_best_match_score_value'] >=0.7)]
print(negtarget_copy_filter.shape)
negtarget_copy_filter['RT_diff'] = abs(negtarget_copy_filter['Average Rt(min)'] - negtarget_copy_filter['RT_neg'])
print(negtarget_copy_filter[negtarget_copy_filter['RT_diff'] <=0.5].shape)
print(tabulate.tabulate(negtarget_copy_filter[negtarget_copy_filter['RT_diff'] <=0.5], headers='keys', tablefmt='psql'))
print(tabulate.tabulate(negtarget_copy_filter[negtarget_copy_filter['RT_diff'].isna()], headers='keys', tablefmt='psql'))

(32, 32)
(1, 33)
+-----+--------------+-------------------+--------------+-----------------+---------------+------------------------------+--------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+-----------------------------+-----------------------------+-------------------+----------------+---------------------+-------------+------------+----------+----------+------------------+------------------------+-----------------+-------------------------------+------------------------+-------------------------+-----------------------------------------------------------------------------------------------------------------+----------------------+------------------------------------+----------------------------+----------------------------------+-----------+
|     |   Unnamed: 0 |   Average Rt(min) |   Average Mz | feature_id      | DTXSID_x      | SMILES_STD                   | PREFERRED_NAME_x   | MOLECULAR_F

In [20]:
def remove_redundant_rows4(df):
    df = df.copy()

    # Ensure numeric columns
    df["library_best_match_score_value"] = pd.to_numeric(
        df.get("library_best_match_score_value", np.nan), errors="coerce"
    )
    if "max_intensity" in df.columns:
        df["max_intensity"] = pd.to_numeric(df["max_intensity"], errors="coerce")

    kept_rows = []

    for dtxsid, sub in df.groupby("DTXSID_x"):
        sub = sub.copy()
        used = set()
        idx_list = list(sub.index)

        for i in idx_list:
            if i in used:
                continue

            row = sub.loc[i]

            # ppm and RT differences vs all rows in this DTXSID group
            ppm = abs(row["Average Mz"] - sub["Average Mz"]) / row["Average Mz"] * 1e6
            rtfilter = abs(row["Average Rt(min)"] - sub["Average Rt(min)"])

            # cluster: rows within ppm/RT thresholds
            cluster_idx = sub.index[(ppm <= 10) & (rtfilter <= 0.6)].tolist()
            cluster = sub.loc[cluster_idx]

            # choose best row in this cluster
            if cluster["library_best_match_score_value"].notna().any():
                best_idx = cluster["library_best_match_score_value"].idxmax()
            elif "max_intensity" in cluster.columns and cluster["max_intensity"].notna().any():
                best_idx = cluster["max_intensity"].idxmax()
            else:
                best_idx = i  # fallback

            kept_rows.append(df.loc[best_idx])
            used.update(cluster_idx)

    filtered_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    return filtered_df

Tmsmsmatch_pos_level1 = remove_redundant_rows4(postarget_copy_filter[postarget_copy_filter['RT_diff'] <=0.5])
Tmsmsmatch_neg_level1 = remove_redundant_rows4(negtarget_copy_filter[negtarget_copy_filter['RT_diff'] <=0.5])

Tmsmsmatch_pos_level2 = remove_redundant_rows4(postarget_copy_filter[postarget_copy_filter['RT_pos'].isna()])
Tmsmsmatch_neg_level2 = remove_redundant_rows4(negtarget_copy_filter[negtarget_copy_filter['RT_neg'].isna()])

#print match results
print(Tmsmsmatch_pos_level1.shape)
print(Tmsmsmatch_pos_level2.shape)
print(Tmsmsmatch_neg_level1.shape)
print(Tmsmsmatch_neg_level2.shape)

(1, 33)
(27, 33)
(2, 33)
(4, 33)


In [ ]:
#get more spectrum from automsms datafiles
#modifye extraction function to get retention time

import math
import numpy as np
import pandas as pd

from pyopenms import (
    MSExperiment,
    MzMLFile,
    SpectrumLookup,
)

# ---------- helper functions ----------

def ppm_diff(theoretical_mz: float, observed_mz: float) -> float:
    """Absolute ppm difference between theoretical and observed m/z."""
    return abs(observed_mz - theoretical_mz) / theoretical_mz * 1e6


def load_experiment(mzml_path: str) -> MSExperiment:
    exp = MSExperiment()
    MzMLFile().load(mzml_path, exp)
    exp.sortSpectra(True)  # sort by RT
    return exp


def extract_ms2_with_precursors(exp: MSExperiment):
    """
    Go through all spectra, collect MS2 spectra with:
      - precursor m/z
      - precursor charge (if available)
      - RT (in minutes)
      - their peak lists

    Returns a list of dicts:
      [
        {
          "index": spectrum_index_in_exp,
          "native_id": spectrum.nativeID,
          "precursor_mz": float,
          "precursor_charge": int or None,
          "rt_min": float,
          "mz_array": np.array,
          "int_array": np.array,
        },
        ...
      ]
    """
    ms2_list = []

    for i, spec in enumerate(exp):
        if spec.getMSLevel() != 2:
            continue

        # retention time (OpenMS stores RT in seconds)
        rt_sec = spec.getRT()
        rt_min = rt_sec / 60.0

        precursor_mz = None
        precursor_charge = None

        precursors = spec.getPrecursors()
        if precursors:
            precursor = precursors[0]
            precursor_mz = precursor.getMZ()
            precursor_charge = precursor.getCharge()

        if precursor_mz is None or precursor_mz == 0:
            # skip spectra without proper precursor info
            continue

        mz_array, int_array = spec.get_peaks()
        mz_array = np.array(mz_array, dtype=float)
        int_array = np.array(int_array, dtype=float)

        ms2_list.append(
            {
                "index": i,
                "native_id": spec.getNativeID(),
                "precursor_mz": float(precursor_mz),
                "precursor_charge": int(precursor_charge) if precursor_charge != 0 else None,
                "rt_min": float(rt_min),
                "mz_array": mz_array,
                "int_array": int_array,
            }
        )

    return ms2_list


def load_candidates(candidate_csv: str,
                    mz_col: str = "Average Mz",
                    rt_col: str = "Average Rt(min)",
                    inchikey_col: str = "INCHIKEY"):
    df = pd.read_csv(candidate_csv)

    for col in [mz_col, rt_col]:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in candidate file.")

    if inchikey_col not in df.columns:
        df[inchikey_col] = ""

    return df, mz_col, rt_col, inchikey_col


def match_candidates_to_ms2(
    ms2_list,
    candidates_df,
    mz_col,
    rt_col,
    inchikey_col,
    ppm_tolerance: float = 10.0,
    rt_tolerance_min: float = 0.5,
):
    """
    For each candidate (Average Mz, Average Rt(min)), find matching MS2 spectra:

      |ppm| <= ppm_tolerance
      |RT diff| <= rt_tolerance_min (minutes)

    Returns a list of dicts with candidate + spectrum info.
    """
    matches = []

    ms2_precursor_mz = np.array([s["precursor_mz"] for s in ms2_list], dtype=float)
    ms2_rt_min = np.array([s["rt_min"] for s in ms2_list], dtype=float)

    for idx, row in candidates_df.iterrows():
        cand_mz = float(row[mz_col])
        cand_rt = float(row[rt_col])
        inchikey = row[inchikey_col]

        # RT pre-filter
        rt_diff = np.abs(ms2_rt_min - cand_rt)
        rt_mask = rt_diff <= rt_tolerance_min
        if not np.any(rt_mask):
            continue

        # ppm filter on that subset
        candidate_mz_subset = ms2_precursor_mz[rt_mask]
        ppm_values = np.abs(candidate_mz_subset - cand_mz) / cand_mz * 1e6
        ppm_mask = ppm_values <= ppm_tolerance

        if not np.any(ppm_mask):
            continue

        # indices in the original ms2_list
        ms2_indices = np.where(rt_mask)[0][ppm_mask]

        for ms2_idx in ms2_indices:
            s = ms2_list[ms2_idx]

            matches.append(
                {
                    "candidate_row": int(idx),
                    "inchikey": inchikey,
                    "candidate_mz": cand_mz,
                    "candidate_rt_min": cand_rt,
                    "spectrum_native_id": s["native_id"],
                    "spectrum_index": s["index"],
                    "precursor_mz": s["precursor_mz"],
                    "precursor_charge": s["precursor_charge"],
                    "rt_min": s["rt_min"],
                    "mz_array": s["mz_array"],
                    "int_array": s["int_array"],
                }
            )

    return matches


from collections import defaultdict
from matchms import Spectrum as MatchMSSpectrum

def build_query_spectra_from_matches(matches, baseline, normalize_and_filter):
    """
    Convert pyOpenMS 'matches' into matchms Spectrum objects.

    Returns:
      dict[inchikey] -> list[matchms.Spectrum]
    """
    query_dict = defaultdict(list)

    for m in matches:
        key = m["inchikey"]  # matches your 'InChiKey_origin' logic
        if not key:
            continue

        mz_raw = m["mz_array"]
        int_raw = m["int_array"]

        # Use your existing preprocessing
        mz_proc, int_proc = normalize_and_filter(
            (mz_raw, int_raw), baseline=baseline
        )

        metadata = {
            "inchikey": key,
            "peak_comments": f"RT={m['rt_min']:.2f}min;nativeID={m['spectrum_native_id']}"
        }

        spectrum = MatchMSSpectrum(
            mz=mz_proc,
            intensities=int_proc,
            metadata=metadata
        )
        query_dict[key].append(spectrum)

    return query_dict


#filepath
autopos_path_r1 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\0802_pool_sample_positive\B11-20pool_MSMS_pos_r1.mzML'
autopos_path_r2 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\0802_pool_sample_positive\B11-20pool_MSMS_pos_r2.mzML'
autoneg_path_r1 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\0802_pool_sample_negative\B11-20pool_MSMS_neg_r1.mzML'
autoneg_path_r2 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\0802_pool_sample_negative\B11-20pool_MSMS_neg_r2.mzML'

exp_neg1 = load_experiment(autoneg_path_r1)
ms2_neg_list1 = extract_ms2_with_precursors(exp_neg1)

matches_neg = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv')
matches_neg_copy = matches_neg.copy()

candidates_df, mz_col, rt_col, inchikey_col = load_candidates(
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv',
    mz_col="Average Mz",
    rt_col="Average Rt(min)",
    inchikey_col="InChiKey_origin",
)

matches = match_candidates_to_ms2(
    ms2_neg_list1,
    candidates_df,
    mz_col,
    rt_col,
    inchikey_col,
    ppm_tolerance=10.0,
    rt_tolerance_min=0.5,
)


import logging
from matchms import Spectrum  # alias if you want, but we used MatchMSSpectrum above

logging.getLogger("matchms").setLevel(logging.ERROR)

# firstmatch_pos is your dataframe of candidates to test.
# It should at least contain a column with the InChIKeys you used above.
# I assume that column is 'InChiKey_origin' as in your snippet.
peak_norm_tol = 0.01  # whatever you used before
query_spectra_by_inchikey = build_query_spectra_from_matches(
    matches, baseline=peak_norm_tol, normalize_and_filter=normalize_and_filter
)

for iter_idx, row in matches_neg_copy.iterrows():
    key = row['InChiKey_origin']

    # --------------- QUERY SPECTRA (experimental) ---------------
    # use spectra built from the AutoMSMS mzML
    query_spectrums = query_spectra_by_inchikey.get(key, [])

    if len(query_spectrums) == 0:
        print(f'No query spectrum found for compound {key}')
        continue

    # --------------- REFERENCE SPECTRA (libraries) ---------------
    reference_spectrums_library = []
    reference_spectrums_insilico = []

    sources = [
        (mbank_data_neg,          reference_spectrums_library, 'mbank'),
        (mona_data_neg,           reference_spectrums_library, 'mona'),
        (mona_obtrap_data_neg,    reference_spectrums_library, 'mona_obtrap'),
        (inhouse_data_neg,        reference_spectrums_library, 'inhouse1'),
        (inhouse_data_neg2,       reference_spectrums_library, 'inhouse2'),
        # (insilico_data_neg,    reference_spectrums_insilico, 'insilico'),  # example
    ]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)  # your existing function

        # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    # --------------- MATCHING (your existing function) ---------------
    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
        reference_spectrums_library,
        query_spectrums,
        match_tolerance,
        key
    )
    # --------------- SAVE RESULTS ---------------
    matches_neg_copy.loc[iter_idx, 'library_best_match'] = best_spectrum_library
    matches_neg_copy.loc[iter_idx, 'library_best_match_num_matchpeak'] = best_num_matchpeak_library
    matches_neg_copy.loc[iter_idx, 'library_best_match_query'] = best_match_query_library
    matches_neg_copy.loc[iter_idx, 'library_best_match_score_value'] = best_score_library


In [25]:
#summary of the matching spectrum
summary_neg_auto1 = matches_neg_copy['library_best_match_score_value'].describe()
print(summary_neg_auto1)

count     1032.00000
unique      88.00000
top          0.99931
freq       116.00000
Name: library_best_match_score_value, dtype: float64


In [26]:
exp_neg2 = load_experiment(autoneg_path_r2)
ms2_neg_list2 = extract_ms2_with_precursors(exp_neg2)

exp_pos = load_experiment(autopos_path_r1)
exp_pos2 = load_experiment(autopos_path_r2)
ms2_pos_list = extract_ms2_with_precursors(exp_pos)
ms2_pos_list2 = extract_ms2_with_precursors(exp_pos2)

candidates_df, mz_col, rt_col, inchikey_col = load_candidates(
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv',
    mz_col="Average Mz",
    rt_col="Average Rt(min)",
    inchikey_col="InChiKey_origin",
)

matches_neg2 = match_candidates_to_ms2(
    ms2_neg_list2,
    candidates_df,
    mz_col,
    rt_col,
    inchikey_col,
    ppm_tolerance=10.0,
    rt_tolerance_min=0.5,
)

candidates_df, mz_col, rt_col, inchikey_col = load_candidates(
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv',
    mz_col="Average Mz",
    rt_col="Average Rt(min)",
    inchikey_col="InChiKey_origin",
)

matches_ms2_pos = match_candidates_to_ms2(
    ms2_pos_list,
    candidates_df,
    mz_col,
    rt_col,
    inchikey_col,
    ppm_tolerance=10.0,
    rt_tolerance_min=0.5, 
)

matches_ms2_pos2 = match_candidates_to_ms2(
    ms2_pos_list2,
    candidates_df,
    mz_col,
    rt_col,
    inchikey_col,
    ppm_tolerance=10.0,
    rt_tolerance_min=0.5, 
)

In [27]:
import logging
from matchms import Spectrum  # alias if you want, but we used MatchMSSpectrum above

logging.getLogger("matchms").setLevel(logging.ERROR)

# firstmatch_pos is your dataframe of candidates to test.
# It should at least contain a column with the InChIKeys you used above.
# I assume that column is 'InChiKey_origin' as in your snippet.
query_spectra_by_inchikey = build_query_spectra_from_matches(
    matches_neg2, baseline=peak_norm_tol, normalize_and_filter=normalize_and_filter
)

for iter_idx, row in matches_neg_copy.iterrows():
    key = row['InChiKey_origin']

    # --------------- QUERY SPECTRA (experimental) ---------------
    # use spectra built from the AutoMSMS mzML
    query_spectrums = query_spectra_by_inchikey.get(key, [])

    if len(query_spectrums) == 0:
        print(f'No query spectrum found for compound {key}')
        continue

    # --------------- REFERENCE SPECTRA (libraries) ---------------
    reference_spectrums_library = []
    reference_spectrums_insilico = []

    sources = [
        (mbank_data_neg,          reference_spectrums_library, 'mbank'),
        (mona_data_neg,           reference_spectrums_library, 'mona'),
        (mona_obtrap_data_neg,    reference_spectrums_library, 'mona_obtrap'),
        (inhouse_data_neg,        reference_spectrums_library, 'inhouse1'),
        (inhouse_data_neg2,       reference_spectrums_library, 'inhouse2'),
        # (insilico_data_neg,    reference_spectrums_insilico, 'insilico'),  # example
    ]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)  # your existing function

        # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    # --------------- MATCHING (your existing function) ---------------
    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
        reference_spectrums_library,
        query_spectrums,
        match_tolerance,
        key
    )
    # --------------- SAVE RESULTS ---------------
    matches_neg_copy.loc[iter_idx, 'library_best_match2'] = best_spectrum_library
    matches_neg_copy.loc[iter_idx, 'library_best_match_num_matchpeak2'] = best_num_matchpeak_library
    matches_neg_copy.loc[iter_idx, 'library_best_match_query2'] = best_match_query_library
    matches_neg_copy.loc[iter_idx, 'library_best_match_score_value2'] = best_score_library


No query spectrum found for compound KDCGOANMDULRCW-UHFFFAOYSA-N
No query spectrum found for compound QUKPALAWEPMWOS-UHFFFAOYSA-N
No query spectrum found for compound IDKAXRLETRCXKS-UHFFFAOYSA-N
No query spectrum found for compound LXFMMUDXRIMBHN-UHFFFAOYSA-N
No query spectrum found for compound XGRJZXREYAXTGV-UHFFFAOYSA-N
No query spectrum found for compound OGFCWEKDPGAFHW-UHFFFAOYSA-N
No query spectrum found for compound PCLGVSQVYVGFSH-UHFFFAOYSA-N
No query spectrum found for compound PMCGHPATMOFOKW-UHFFFAOYSA-N
No query spectrum found for compound IBWBDNBSIFGSLW-UHFFFAOYSA-N
No query spectrum found for compound VUQVJIUBUPPCDB-UHFFFAOYSA-N
No query spectrum found for compound VPRAZXISONXJGO-UHFFFAOYSA-N
No query spectrum found for compound GWXAPUCCUTUZCC-UHFFFAOYSA-N
No query spectrum found for compound VVSCJSQUDDOXNL-UHFFFAOYSA-N
No query spectrum found for compound FVJDXDLHPOTFLL-UHFFFAOYSA-N
No query spectrum found for compound QUVSFZCWYBQJJK-UHFFFAOYSA-N
No query spectrum found f

In [28]:
#positive spectrum
matches_pos = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv')
matches_pos_copy = matches_pos.copy()

query_spectra_by_inchikey = build_query_spectra_from_matches(
    matches_ms2_pos, baseline=peak_norm_tol, normalize_and_filter=normalize_and_filter
)

for iter_idx, row in matches_pos_copy.iterrows():
    key = row['InChiKey_origin']

    # --------------- QUERY SPECTRA (experimental) ---------------
    # use spectra built from the AutoMSMS mzML
    query_spectrums = query_spectra_by_inchikey.get(key, [])

    if len(query_spectrums) == 0:
        print(f'No query spectrum found for compound {key}')
        continue

    # --------------- REFERENCE SPECTRA (libraries) ---------------
    reference_spectrums_library = []
    reference_spectrums_insilico = []

    sources = [
        (mbank_data_pos,          reference_spectrums_library, 'mbank'),
        (mona_data_pos,           reference_spectrums_library, 'mona'),
        (mona_obtrap_data_pos,    reference_spectrums_library, 'mona_obtrap'),
        (inhouse_data_pos,        reference_spectrums_library, 'inhouse1'),
        (inhouse_data_pos2,       reference_spectrums_library, 'inhouse2'),
        # (insilico_data_neg,    reference_spectrums_insilico, 'insilico'),  # example
    ]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)  # your existing function

        # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    # --------------- MATCHING (your existing function) ---------------
    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
        reference_spectrums_library,
        query_spectrums,
        match_tolerance,
        key
    )
    # --------------- SAVE RESULTS ---------------
    matches_pos_copy.loc[iter_idx, 'library_best_match'] = best_spectrum_library
    matches_pos_copy.loc[iter_idx, 'library_best_match_num_matchpeak'] = best_num_matchpeak_library
    matches_pos_copy.loc[iter_idx, 'library_best_match_query'] = best_match_query_library
    matches_pos_copy.loc[iter_idx, 'library_best_match_score_value'] = best_score_library


No query spectrum found for compound IRYJRGCIQBGHIV-UHFFFAOYSA-N
No query spectrum found for compound MGHNWMRNFGYJKE-UHFFFAOYSA-N
No query spectrum found for compound HQGPKMSGXAUKHT-UHFFFAOYSA-N
No query spectrum found for compound WMAHFAYLCMTHCB-UHFFFAOYSA-N
No query spectrum found for compound OMZKZKGVFMWHOU-UHFFFAOYSA-N
No query spectrum found for compound DCWJVLCAHFTROS-UHFFFAOYSA-N
No query spectrum found for compound QNMBSXGYAQZCTN-UHFFFAOYSA-N
No query spectrum found for compound YACJUWJRBSAQIG-UHFFFAOYSA-N
No query spectrum found for compound GTZCVFVGUGFEME-UHFFFAOYSA-N
No query spectrum found for compound VWPUAXALDFFXJW-UHFFFAOYSA-N
No query spectrum found for compound BQDSJBHEOWAFTE-UHFFFAOYSA-N
No query spectrum found for compound GSGMKOPCDJFHEF-UHFFFAOYSA-N
No query spectrum found for compound ALCDAWARCQFJBA-UHFFFAOYSA-N
No query spectrum found for compound DJJPSSQMPHHRHD-UHFFFAOYSA-N
No query spectrum found for compound KRKNYBCHXYNGOX-UHFFFAOYSA-N
No query spectrum found f

In [32]:
summary_pos_auto1 = matches_pos_copy['library_best_match_score_value'].describe()
print(summary_pos_auto1)

count     1608.000000
unique     121.000000
top          0.998303
freq       123.000000
Name: library_best_match_score_value, dtype: float64


In [33]:
query_spectra_by_inchikey = build_query_spectra_from_matches(
    matches_ms2_pos2, baseline=peak_norm_tol, normalize_and_filter=normalize_and_filter
)

for iter_idx, row in matches_pos_copy.iterrows():
    key = row['InChiKey_origin']

    # --------------- QUERY SPECTRA (experimental) ---------------
    # use spectra built from the AutoMSMS mzML
    query_spectrums = query_spectra_by_inchikey.get(key, [])

    if len(query_spectrums) == 0:
        print(f'No query spectrum found for compound {key}')
        continue

    # --------------- REFERENCE SPECTRA (libraries) ---------------
    reference_spectrums_library = []
    reference_spectrums_insilico = []

    sources = [
        (mbank_data_pos,          reference_spectrums_library, 'mbank'),
        (mona_data_pos,           reference_spectrums_library, 'mona'),
        (mona_obtrap_data_pos,    reference_spectrums_library, 'mona_obtrap'),
        (inhouse_data_pos,        reference_spectrums_library, 'inhouse1'),
        (inhouse_data_pos2,       reference_spectrums_library, 'inhouse2'),
        # (insilico_data_neg,    reference_spectrums_insilico, 'insilico'),  # example
    ]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)  # your existing function

        # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    # --------------- MATCHING (your existing function) ---------------
    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
        reference_spectrums_library,
        query_spectrums,
        match_tolerance,
        key
    )
    # --------------- SAVE RESULTS ---------------
    matches_pos_copy.loc[iter_idx, 'library_best_match2'] = best_spectrum_library
    matches_pos_copy.loc[iter_idx, 'library_best_match_num_matchpeak2'] = best_num_matchpeak_library
    matches_pos_copy.loc[iter_idx, 'library_best_match_query2'] = best_match_query_library
    matches_pos_copy.loc[iter_idx, 'library_best_match_score_value2'] = best_score_library


No query spectrum found for compound IRYJRGCIQBGHIV-UHFFFAOYSA-N
No query spectrum found for compound MGHNWMRNFGYJKE-UHFFFAOYSA-N
No query spectrum found for compound HQGPKMSGXAUKHT-UHFFFAOYSA-N
No query spectrum found for compound WMAHFAYLCMTHCB-UHFFFAOYSA-N
No query spectrum found for compound OMZKZKGVFMWHOU-UHFFFAOYSA-N
No query spectrum found for compound DCWJVLCAHFTROS-UHFFFAOYSA-N
No query spectrum found for compound QNMBSXGYAQZCTN-UHFFFAOYSA-N
No query spectrum found for compound GTZCVFVGUGFEME-UHFFFAOYSA-N
No query spectrum found for compound VWPUAXALDFFXJW-UHFFFAOYSA-N
No query spectrum found for compound BQDSJBHEOWAFTE-UHFFFAOYSA-N
No query spectrum found for compound GSGMKOPCDJFHEF-UHFFFAOYSA-N
No query spectrum found for compound ALCDAWARCQFJBA-UHFFFAOYSA-N
No query spectrum found for compound DJJPSSQMPHHRHD-UHFFFAOYSA-N
No query spectrum found for compound KRKNYBCHXYNGOX-UHFFFAOYSA-N
No query spectrum found for compound ANEDZEVDORCLPM-UHFFFAOYSA-N
No query spectrum found f

In [35]:
#combine two files and make summary
print(matches_pos_copy.shape)

(36300, 35)


In [60]:
#combine column of library_best_match2, library_best_match_num_matchpeak2, library_best_match_query2, library_best_match_score_value2, and library_best_match, library_best_match_num_matchpeak, library_best_match_query, library_best_match_score_value, whichever has the not null value and large value, make it as the final matching socres
def combine_library_matches(df):
    """
    Combine AutoMSMS- and AIF-based library matching results.
    
    Rules:
        - Pick the entry with the highest non-null score among:
            library_best_match_score_value
            library_best_match_score_value2
        - Copy the corresponding match info (match, n_peaks, query).
    
    New output columns:
        final_best_match
        final_best_num_matchpeak
        final_best_match_query
        final_best_match_score_value
    """

    # ensure numeric scores
    df['library_best_match_score_value']  = pd.to_numeric(df['library_best_match_score_value'], errors="coerce")
    df['library_best_match_score_value2'] = pd.to_numeric(df['library_best_match_score_value2'], errors="coerce")

    # choose the best score
    df['final_best_match_score_value'] = df[['library_best_match_score_value',
                                             'library_best_match_score_value2']].max(axis=1)

    # boolean mask for which score was chosen
    mask_use_set1 = df['final_best_match_score_value'] == df['library_best_match_score_value']
    mask_use_set2 = df['final_best_match_score_value'] == df['library_best_match_score_value2']

    # initialize final output columns
    df['final_best_match'] = None
    df['final_best_num_matchpeak'] = None
    df['final_best_match_query'] = None

    # fill from set 1
    df.loc[mask_use_set1, 'final_best_match']         = df.loc[mask_use_set1, 'library_best_match']
    df.loc[mask_use_set1, 'final_best_num_matchpeak'] = df.loc[mask_use_set1, 'library_best_match_num_matchpeak']
    df.loc[mask_use_set1, 'final_best_match_query']   = df.loc[mask_use_set1, 'library_best_match_query']

    # fill from set 2
    df.loc[mask_use_set2, 'final_best_match']         = df.loc[mask_use_set2, 'library_best_match2']
    df.loc[mask_use_set2, 'final_best_num_matchpeak'] = df.loc[mask_use_set2, 'library_best_match_num_matchpeak2']
    df.loc[mask_use_set2, 'final_best_match_query']   = df.loc[mask_use_set2, 'library_best_match_query2']

    return df

#filter rows with none value in library_best_match_score_value, and library_best_match_score_value2    
matches_pos_copy2 = combine_library_matches(matches_pos_copy)
matches_neg_copy2 = combine_library_matches(matches_neg_copy)

#filter rows with final_best_num_matchpeak >=3, and final_best_num_matchpeak >=0.5
matches_pos_copy2_filter = matches_pos_copy2[(matches_pos_copy2['final_best_num_matchpeak']>=2)&(matches_pos_copy2['final_best_match_score_value']>=0.7)]
matches_neg_copy2_filter = matches_neg_copy2[(matches_neg_copy2['final_best_num_matchpeak']>=2)&(matches_neg_copy2['final_best_match_score_value']>=0.7)]

print(matches_pos_copy2_filter.shape)
print(matches_neg_copy2_filter.shape)

(760, 39)
(534, 39)


In [75]:
#output matches from automsms 
matches_pos_copy2['matching_by'] = 'dda'
matches_neg_copy2['matching_by'] = 'dda'
matches_pos_copy2.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\positive_matches_dda_from0802.csv')
matches_neg_copy2.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\negative_matches_dda_from0802.csv')

In [ ]:
#make spectrum plot using mirror plot, make ms1 EIC plot with LC gradient


In [61]:
#get the number of unique DTXSID
print(len(matches_pos_copy2_filter['InChiKey_origin'].unique()))
print(len(matches_neg_copy2_filter['InChiKey_origin'].unique()))

#also get RTdiff
matches_pos_copy2_filter['RTdiff'] = abs(matches_pos_copy2_filter['Average Rt(min)']-matches_pos_copy2_filter['RT_pos'])
matches_neg_copy2_filter['RTdiff'] = abs(matches_neg_copy2_filter['Average Rt(min)']- matches_neg_copy2_filter['RT_neg'])

#filter rows with RTdiff <0.5
matches_pos_level1 = matches_pos_copy2_filter[matches_pos_copy2_filter['RTdiff']<=0.5]
matches_neg_level1 = matches_neg_copy2_filter[matches_neg_copy2_filter['RTdiff'] <= 0.5]
print(len(matches_pos_level1['InChiKey_origin'].unique()))
print(len(matches_neg_level1['InChiKey_origin'].unique()))

#filter row with NA value in RT_pos or RT_neg
matches_pos_level2 = matches_pos_copy2_filter[matches_pos_copy2_filter['RT_pos'].isna()]
matches_neg_level2 = matches_neg_copy2_filter[matches_neg_copy2_filter['RT_neg'].isna()]

54
44
8
7


In [62]:
def remove_redundant_rows3(df):
    df = df.copy()

    # Ensure numeric columns
    df["final_best_match_score_value"] = pd.to_numeric(
        df.get("final_best_match_score_value", np.nan), errors="coerce"
    )
    if "max_intensity" in df.columns:
        df["max_intensity"] = pd.to_numeric(df["max_intensity"], errors="coerce")

    kept_rows = []

    for dtxsid, sub in df.groupby("DTXSID_x"):
        sub = sub.copy()
        used = set()
        idx_list = list(sub.index)

        for i in idx_list:
            if i in used:
                continue

            row = sub.loc[i]

            # ppm and RT differences vs all rows in this DTXSID group
            ppm = abs(row["Average Mz"] - sub["Average Mz"]) / row["Average Mz"] * 1e6
            rtfilter = abs(row["Average Rt(min)"] - sub["Average Rt(min)"])

            # cluster: rows within ppm/RT thresholds
            cluster_idx = sub.index[(ppm <= 10) & (rtfilter <= 0.6)].tolist()
            cluster = sub.loc[cluster_idx]

            # choose best row in this cluster
            if cluster["final_best_match_score_value"].notna().any():
                best_idx = cluster["final_best_match_score_value"].idxmax()
            elif "max_intensity" in cluster.columns and cluster["max_intensity"].notna().any():
                best_idx = cluster["max_intensity"].idxmax()
            else:
                best_idx = i  # fallback

            kept_rows.append(df.loc[best_idx])
            used.update(cluster_idx)

    filtered_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    return filtered_df


automatches_pos_level1 = remove_redundant_rows3(matches_pos_level1)
automatches_neg_level1 = remove_redundant_rows3(matches_neg_level1)

automatches_pos_level2 = remove_redundant_rows3(matches_pos_level2)
automatches_neg_level2 = remove_redundant_rows3(matches_neg_level2)

print(tabulate.tabulate(automatches_pos_level1, headers ='keys', tablefmt='psql'))
print(tabulate.tabulate(automatches_neg_level1, headers = 'keys', tablefmt = 'psql'))

+----+--------------+-------------------+--------------+------------------+---------------+-------------------------------------------+---------------------------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+-----------------------------+-----------------------------+-------------------+----------------+---------------------+-------------+---------------+----------+----------+------------------+------------------------+------------------+-------------------------------+------------------------+-------------------------+---------------------------------+------------------------------------+-------------------------------------+----------------------------------+--------------------------+-------------------------------------+-------------------------------------+-----------------------------------+--------------------------------+---------------------------------+----------------------------+--------

In [63]:
#print unique DTXSID_x in automatches_pos and neg
print(len(automatches_pos_level2['DTXSID_x'].unique()))
print(len(automatches_neg_level2['DTXSID_x'].unique()))

46
37


In [ ]:
#combined the automsmatches, and targetedms matches。
#check common founding by search rows with same DTXSID_x and feature_id
Tmsmsmatch_neg_level1['polarity'] = 'neg'
Tmsmsmatch_pos_level1['polarity'] = 'pos'
Tmsmsmatch_neg_level1['spectrum_acquiredby'] = 'Targeted'
Tmsmsmatch_pos_level1['spectrum_acquiredby'] = 'Targeted'
automatches_neg_level1['polarity'] = 'neg'
automatches_pos_level1['polarity'] = 'pos'
automatches_neg_level1['spectrum_acquiredby'] = 'dda'
automatches_pos_level1['spectrum_acquiredby'] = 'dda'

all_pos_level1 = pd.concat([Tmsmsmatch_pos_level1, automatches_pos_level1], axis=0)
all_neg_level1 = pd.concat([Tmsmsmatch_neg_level1, automatches_neg_level1], axis=0)

all_pos_level2 = pd.concat([Tmsmsmatch_pos_level2, automatches_pos_level2], axis = 0)
all_neg_level2 = pd.concat([Tmsmsmatch_neg_level2, automatches_neg_level2], axis = 0)

print(f'shape of all pos level1:{all_pos_level1.shape}')
print(f'shape of all pos level2:{all_pos_level2.shape}')
print(len(all_pos_level1['DTXSID_x'].unique()))
print(len(all_neg_level1['DTXSID_x'].unique()))


shape of all pos level1:(13, 44)
shape of all pos level2:(251, 42)
8
7


In [65]:
print(len(all_pos_level2['DTXSID_x'].unique()))
print(len(all_neg_level2['DTXSID_x'].unique()))

57
37


In [66]:
print(tabulate.tabulate(all_pos_level1,headers='keys', tablefmt='psql'))
print(tabulate.tabulate(all_neg_level1, headers='keys', tablefmt='psql'))

combinelst = all_pos_level1['DTXSID_x'] +all_neg_level1['DTXSID_x']
print(len(combinelst.unique()))

+----+--------------+-------------------+--------------+------------------+---------------+-------------------------------------------+---------------------------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+-----------------------------+-----------------------------+-------------------+----------------+---------------------+-------------+---------------+----------+----------+------------------+------------------------+------------------+-------------------------------+------------------------+-------------------------+-----------------------------------------------------------------------------------------------------------------+---------------------------------+------------------------------------+-------------------------------------+----------------------------------+-----------+------------+-----------------------+--------------------------+-------------------------------------+---------------

In [ ]:
#push this to google drive
output_path = r"G:\My Drive\REVEAL_UCSF_project\First400\results_pos_level1.csv"
output_path2 = r'G:\My Drive\REVEAL_UCSF_project\First400\results_neg_level1.csv'
all_pos_level1.to_csv(output_path, index=False)
all_neg_level1.to_csv(output_path2, index=False)

In [116]:
#combined the matches from previous 200 samples. by checking the annotation level1 and 2
batch10_l1 = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\level1_annotation_batch1_10.csv')
print(tabulate.tabulate(batch10_l1, headers='keys', tablefmt='psql'))
print(len(batch10_l1['DTXSID'].unique()))

+----+-------------------+--------------+--------------------------------------------------------------------------------+------------------------------+---------------+----------------+-----------+--------------+------------------------+--------------------+-----------------------------+------------------------------+---------------------------------------------------------------------+---------------------------+---------------+---------------------+---------------------+---------------+-------------------+--------------------------+------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------+-----------------+--------------------+
|    |   Average Rt(min) |   Average Mz | Matched Compound                                                               | PREFERRED_NAME               | DTXSID        |   RT_reference |   RT_diff |   best_score |   best_match_num_p

In [ ]:
#combine level 2 compounds and make summary

In [ ]:
# #using previous acquired msms for more spectrum matching
# #dda spectrum from bath10
# # b10dda_pos1 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_dda\B1-10pool_AutoMSMS_pos_r1a.mzML'
# # b10dda_pos2 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_dda\B1-10pool_AutoMSMS_pos_r2a.mzML'
# # b10dda_neg1= r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_dda\B1-10pool_AutoMSMS_Neg_r1a.mzML'
# # b10dda_neg2= r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_dda\B1-10pool_AutoMSMS_Neg_r2a.mzML'

# matches_neg = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv')
# b10_matches_neg_copy = matches_neg.copy()

# b10ta_neg1=r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I1.mzML'
# b10ta_neg2= r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I2.mzML'
# b10ta_neg3 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I3.mzML'
# b10ta_neg4 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I4.mzML'
# b10ta_neg5 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I5.mzML'
# b10ta_neg6 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I6.mzML'
# b10ta_neg7 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I7.mzML'
# b10ta_neg8 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I8.mzML'
# b10ta_neg9 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I9.mzML'
# b10ta_neg10 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I10.mzML'
# b10ta_neg11 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I11.mzML'
# b10ta_neg12 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I12.mzML'
# b10ta_neg13 = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I13.mzML'

# b10_exp_neg1 = load_experiment(b10ta_neg1)
# b10_ms2_neg_list1 = extract_ms2_with_precursors(b10_exp_neg1)
# b10_exp_neg2 = load_experiment(b10ta_neg2)
# b10_ms2_neg_list2 = extract_ms2_with_precursors(b10_exp_neg2)
# b10_exp_neg3 = load_experiment(b10ta_neg3)
# b10_ms2_neg_list3 = extract_ms2_with_precursors(b10_exp_neg3)
# b10_exp_neg4 = load_experiment(b10ta_neg4)
# b10_ms2_neg_list4 = extract_ms2_with_precursors(b10_exp_neg4)
# b10_exp_neg5 = load_experiment(b10ta_neg5)
# b10_ms2_neg_list5 = extract_ms2_with_precursors(b10_exp_neg5)
# b10_exp_neg6 = load_experiment(b10ta_neg6)
# b10_ms2_neg_list6 = extract_ms2_with_precursors(b10_exp_neg6)
# b10_exp_neg7 = load_experiment(b10ta_neg7)
# b10_ms2_neg_list7 = extract_ms2_with_precursors(b10_exp_neg7)
# b10_exp_neg8 = load_experiment(b10ta_neg8)
# b10_ms2_neg_list8 = extract_ms2_with_precursors(b10_exp_neg8)
# b10_exp_neg9 = load_experiment(b10ta_neg9)
# b10_ms2_neg_list9 = extract_ms2_with_precursors(b10_exp_neg9)
# b10_exp_neg10 = load_experiment(b10ta_neg10)
# b10_ms2_neg_list10 = extract_ms2_with_precursors(b10_exp_neg10)
# b10_exp_neg10 = load_experiment(b10ta_neg10)
# b10_ms2_neg_list10 = extract_ms2_with_precursors(b10_exp_neg10)
# b10_exp_neg11 = load_experiment(b10ta_neg11)
# b10_ms2_neg_list11 = extract_ms2_with_precursors(b10_exp_neg11)
# b10_exp_neg12 = load_experiment(b10ta_neg12)
# b10_ms2_neg_list12 = extract_ms2_with_precursors(b10_exp_neg12)
# b10_exp_neg13 = load_experiment(b10ta_neg13)
# b10_ms2_neg_list13 = extract_ms2_with_precursors(b10_exp_neg13)


# b10_candidates_df, b10_mz_col, b10_rt_col, b10_inchikey_col = load_candidates(
#     r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv',
#     mz_col="Average Mz",
#     rt_col="Average Rt(min)",
#     inchikey_col="InChiKey_origin",
# )

# b10_matches = match_candidates_to_ms2(
#     b10_ms2_neg_list1,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )

# b10_matches2 = match_candidates_to_ms2(
#     b10_ms2_neg_list2,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )

# b10_matches3 = match_candidates_to_ms2(
#     b10_ms2_neg_list3,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )

# b10_matches4 = match_candidates_to_ms2(
#     b10_ms2_neg_list4,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches5 = match_candidates_to_ms2(
#     b10_ms2_neg_list5,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches6 = match_candidates_to_ms2(
#     b10_ms2_neg_list6,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches7 = match_candidates_to_ms2(
#     b10_ms2_neg_list7,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches8 = match_candidates_to_ms2(
#     b10_ms2_neg_list8,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches9 = match_candidates_to_ms2(
#     b10_ms2_neg_list9,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches10 = match_candidates_to_ms2(
#     b10_ms2_neg_list10,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches11 = match_candidates_to_ms2(
#     b10_ms2_neg_list11,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches12 = match_candidates_to_ms2(
#     b10_ms2_neg_list12,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )
# b10_matches13 = match_candidates_to_ms2(
#     b10_ms2_neg_list13,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )

In [42]:
import pandas as pd

# ---------------------------------------------------------------
# 1. Load matched peak table (negative mode) and make a working copy
# ---------------------------------------------------------------
matches_neg_path = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv'

matches_neg = pd.read_csv(matches_neg_path)
b10_matches_neg_copy = matches_neg.copy()

# ---------------------------------------------------------------
# 2. Targeted MS/MS mzML files (batch 1–10 negative targeted MSMS)
# ---------------------------------------------------------------
b10ta_neg_files = [
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I1.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I2.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I3.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I4.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I5.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I6.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I7.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I8.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_targetmsms\B1-10-MSMS-neg_I9.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_msmsdata2\REVEAL_B1-10pool_Neg_Inj1.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_msmsdata2\REVEAL_B1-10pool_Neg_Inj2.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_msmsdata2\REVEAL_B1-10pool_Neg_Inj3.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\negative_msmsdata2\REVEAL_B1-10pool_Neg_Inj4.mzML'
]

# ---------------------------------------------------------------
# 3. Load each mzML and extract its MS2 with precursor info
# ---------------------------------------------------------------
b10_ms2_neg_lists = []
for fpath in b10ta_neg_files:
    exp = load_experiment(fpath)
    ms2_list = extract_ms2_with_precursors(exp)
    b10_ms2_neg_lists.append(ms2_list)

# ---------------------------------------------------------------
# 4. Load candidate table (MSDIAL-style): Average Mz, Average Rt(min), InChiKey_origin
# ---------------------------------------------------------------
b10_candidates_df, b10_mz_col, b10_rt_col, b10_inchikey_col = load_candidates(
    matches_neg_path,
    mz_col="Average Mz",
    rt_col="Average Rt(min)",
    inchikey_col="InChiKey_origin",
)

# ---------------------------------------------------------------
# 5. Match candidates to MS2 for each targeted run (13 injections)
# ---------------------------------------------------------------
b10_match_sets = []
for ms2_list in b10_ms2_neg_lists:
    matches_for_run = match_candidates_to_ms2(
        ms2_list,
        b10_candidates_df,
        b10_mz_col,
        b10_rt_col,
        b10_inchikey_col,
        ppm_tolerance=10.0,
        rt_tolerance_min=0.5,
    )
    b10_match_sets.append(matches_for_run)

# ---------------------------------------------------------------
# 6. Build query spectra (matchms) from all match sets
# ---------------------------------------------------------------
peak_norm_tol = 0.01

b10_query_sets = [
    build_query_spectra_from_matches(
        m,
        baseline=peak_norm_tol,
        normalize_and_filter=normalize_and_filter,
    )
    for m in b10_match_sets
]
# b10_query_sets is a list of dicts: index 0..12, each dict: inchikey -> [Spectrum, ...]

# ---------------------------------------------------------------
# 7. For each candidate row, build reference library spectra and
#    run spectrum matching for all 13 targeted injections
# ---------------------------------------------------------------
for iter_idx, row in b10_matches_neg_copy.iterrows():
    key = row["InChiKey_origin"]

    # --------------- REFERENCE SPECTRA (libraries) ---------------
    reference_spectrums_library = []

    sources = [
        (mbank_data_neg,       reference_spectrums_library, "mbank"),
        (mona_data_neg,        reference_spectrums_library, "mona"),
        (mona_obtrap_data_neg, reference_spectrums_library, "mona_obtrap"),
        (inhouse_data_neg,     reference_spectrums_library, "inhouse1"),
        (inhouse_data_neg2,    reference_spectrums_library, "inhouse2"),
    ]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)

        # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            "inchikey": key,
                            "peak_comments": f"{prefix}_{ce}",
                        },
                    )
                )

    # --------------- MATCHING FOR ALL 13 TARGETED RUNS ---------------
    for i, qs_dict in enumerate(b10_query_sets, start=1):
        query_spectrums = qs_dict.get(key, [])

        if not query_spectrums:
            best_score = None
            best_npeaks = None
            best_lib = None
            best_query = None
        else:
            best_score, best_npeaks, best_lib, best_query = find_best_match(
                reference_spectrums_library,
                query_spectrums,
                match_tolerance,
                key,
            )

        # column suffix: first run = "_b10", later runs = "_b10_2", "_b10_3", ...
        suffix = "" if i == 1 else f"_{i}"

        b10_matches_neg_copy.loc[iter_idx, f"library_best_match_b10{suffix}"] = best_lib
        b10_matches_neg_copy.loc[iter_idx, f"library_best_match_num_matchpeak_b10{suffix}"] = best_npeaks
        b10_matches_neg_copy.loc[iter_idx, f"library_best_match_query_b10{suffix}"] = best_query
        b10_matches_neg_copy.loc[iter_idx, f"library_best_match_score_value_b10{suffix}"] = best_score

In [43]:
#rank the spectrum matching results and take the most confident one as the final output for each row
#rank the library_best_match_score_value and library_best_match_num_matchpeak_b10{suffix}
#among all library_best_match_score_value with library_best_match_num_matchpeak_b10{suffix} >=3, rank the value and take the highest score as the final output, as library_best_score_value for the current row. if no score value with matchpeeak >=3, then rank the rest of the score value, then save the highest score and the library_best_num_matchpeak_b10 accordingly
#if all library_best_match_score_value are NA value, then make it NA value.
import numpy as np
import pandas as pd

def select_final_library_match_b10(df):
    """
    For each row, select the most confident library match among
    all columns:
        library_best_match_score_value_b10, library_best_match_score_value_b10_2, ...
        library_best_match_num_matchpeak_b10, library_best_match_num_matchpeak_b10_2, ...
        library_best_match_b10, library_best_match_b10_2, ...
        library_best_match_query_b10, library_best_match_query_b10_2, ...

    Logic:
        1) Among matches with num_matchpeak >= 3, choose the one with highest score.
        2) If none have num_matchpeak >= 3, choose highest score among all.
        3) If all scores are NaN, final outputs are NaN.
    """

    score_prefix = "library_best_match_score_value_b10"
    npeaks_prefix = "library_best_match_num_matchpeak_b10"
    spec_prefix = "library_best_match_b10"
    query_prefix = "library_best_match_query_b10"

    # --- find all score columns and corresponding suffixes ---
    score_cols = [c for c in df.columns if c.startswith(score_prefix)]
    if not score_cols:
        raise ValueError("No score columns found with prefix 'library_best_match_score_value_b10'.")

    # suffix: '' for main, '_2', '_3', ...
    suffixes = [c[len(score_prefix):] for c in score_cols]

    # sort suffixes in a stable numeric way: '', '_2', '_3', ...
    def suffix_key(s):
        if s == "":
            return 0
        try:
            return int(s[1:])
        except ValueError:
            return 9999

    suffixes = sorted(set(suffixes), key=suffix_key)

    # ensure numeric for scores and num_matchpeak
    for s in suffixes:
        sc_col = f"{score_prefix}{s}"
        np_col = f"{npeaks_prefix}{s}"
        if sc_col in df.columns:
            df[sc_col] = pd.to_numeric(df[sc_col], errors="coerce")
        if np_col in df.columns:
            df[np_col] = pd.to_numeric(df[np_col], errors="coerce")

    # --- row-wise selection function ---
    def pick_best(row):
        candidates = []

        for s in suffixes:
            sc_col = f"{score_prefix}{s}"
            np_col = f"{npeaks_prefix}{s}"
            sp_col = f"{spec_prefix}{s}"
            q_col  = f"{query_prefix}{s}"

            if sc_col not in row.index:
                continue

            score = row[sc_col]
            if pd.isna(score):
                continue  # skip NaN scores entirely

            n_peaks = row[np_col] if np_col in row.index else np.nan
            spec    = row[sp_col] if sp_col in row.index else None
            query   = row[q_col]  if q_col in row.index else None

            candidates.append({
                "suffix": s,
                "score": float(score),
                "n_peaks": float(n_peaks) if not pd.isna(n_peaks) else np.nan,
                "spec": spec,
                "query": query,
            })

        # no non-NaN scores at all
        if not candidates:
            return pd.Series({
                "library_best_match_b10_final": np.nan,
                "library_best_match_num_matchpeak_b10_final": np.nan,
                "library_best_match_query_b10_final": np.nan,
                "library_best_match_score_value_b10_final": np.nan,
                "library_best_match_run_suffix_b10_final": np.nan,
            })

        # 1) filter by n_peaks >= 3 if any
        with_peaks = [c for c in candidates if (not pd.isna(c["n_peaks"]) and c["n_peaks"] >= 3)]
        if with_peaks:
            best = max(with_peaks, key=lambda x: x["score"])
        else:
            # 2) otherwise use highest score among all candidates
            best = max(candidates, key=lambda x: x["score"])

        return pd.Series({
            "library_best_match_b10_final": best["spec"],
            "library_best_match_num_matchpeak_b10_final": best["n_peaks"],
            "library_best_match_query_b10_final": best["query"],
            "library_best_match_score_value_b10_final": best["score"],
            "library_best_match_run_suffix_b10_final": best["suffix"],  # e.g. '', '_2', '_3'
        })

    # --- apply row-wise and merge back ---
    final_df = df.apply(pick_best, axis=1)
    df = pd.concat([df, final_df], axis=1)
    return df


b10_matches_neg_curated = select_final_library_match_b10(b10_matches_neg_copy)

In [44]:
import pandas as pd

# ---------------------------------------------------------------
# 1. Load matched peak table (negative mode) and make a working copy
# ---------------------------------------------------------------
matches_pos_path = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv'

matches_pos = pd.read_csv(matches_pos_path)
b10_matches_pos_copy = matches_pos.copy()

# ---------------------------------------------------------------
# 2. Targeted MS/MS mzML files (batch 1–10 negative targeted MSMS)
# ---------------------------------------------------------------
b10ta_pos_files = [
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_targetmsms\B1-10-MSMS-pos_I1.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_targetmsms\B1-10-MSMS-pos_I2.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_targetmsms\B1-10-MSMS-pos_I3.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_targetmsms\B1-10-MSMS-pos_I4.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_targetmsms\B1-10-MSMS-pos_I5.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_targetmsms\B1-10-MSMS-pos_I6.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_msmsdata2\REVEAL_B1-10pool_Pos_Inj1.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_msmsdata2\REVEAL_B1-10pool_Pos_Inj2.mzML',
    r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\batch1_10\positive_msmsdata2\REVEAL_B1-10pool_Pos_Inj3.mzML'
]

# ---------------------------------------------------------------
# 3. Load each mzML and extract its MS2 with precursor info
# ---------------------------------------------------------------
b10_ms2_pos_lists = []
for fpath in b10ta_pos_files:
    exp = load_experiment(fpath)
    ms2_list = extract_ms2_with_precursors(exp)
    b10_ms2_pos_lists.append(ms2_list)

# ---------------------------------------------------------------
# 4. Load candidate table (MSDIAL-style): Average Mz, Average Rt(min), InChiKey_origin
# ---------------------------------------------------------------
b10_candidates_df, b10_mz_col, b10_rt_col, b10_inchikey_col = load_candidates(
    matches_pos_path,
    mz_col="Average Mz",
    rt_col="Average Rt(min)",
    inchikey_col="InChiKey_origin",
)

# ---------------------------------------------------------------
# 5. Match candidates to MS2 for each targeted run (13 injections)
# ---------------------------------------------------------------
b10_match_sets = []
for ms2_list in b10_ms2_pos_lists:
    matches_for_run = match_candidates_to_ms2(
        ms2_list,
        b10_candidates_df,
        b10_mz_col,
        b10_rt_col,
        b10_inchikey_col,
        ppm_tolerance=10.0,
        rt_tolerance_min=0.5,
    )
    b10_match_sets.append(matches_for_run)

# ---------------------------------------------------------------
# 6. Build query spectra (matchms) from all match sets
# ---------------------------------------------------------------
peak_norm_tol = 0.01

b10_query_sets = [
    build_query_spectra_from_matches(
        m,
        baseline=peak_norm_tol,
        normalize_and_filter=normalize_and_filter,
    )
    for m in b10_match_sets
]
# b10_query_sets is a list of dicts: index 0..12, each dict: inchikey -> [Spectrum, ...]

# ---------------------------------------------------------------
# 7. For each candidate row, build reference library spectra and
#    run spectrum matching for all 13 targeted injections
# ---------------------------------------------------------------
for iter_idx, row in b10_matches_pos_copy.iterrows():
    key = row["InChiKey_origin"]

    # --------------- REFERENCE SPECTRA (libraries) ---------------
    reference_spectrums_library = []

    sources = [
        (mbank_data_pos,       reference_spectrums_library, "mbank"),
        (mona_data_pos,        reference_spectrums_library, "mona"),
        (mona_obtrap_data_pos, reference_spectrums_library, "mona_obtrap"),
        (inhouse_data_pos,     reference_spectrums_library, "inhouse1"),
        (inhouse_data_pos2,    reference_spectrums_library, "inhouse2"),
    ]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)

        # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            "inchikey": key,
                            "peak_comments": f"{prefix}_{ce}",
                        },
                    )
                )

    # --------------- MATCHING FOR ALL 13 TARGETED RUNS ---------------
    for i, qs_dict in enumerate(b10_query_sets, start=1):
        query_spectrums = qs_dict.get(key, [])

        if not query_spectrums:
            best_score = None
            best_npeaks = None
            best_lib = None
            best_query = None
        else:
            best_score, best_npeaks, best_lib, best_query = find_best_match(
                reference_spectrums_library,
                query_spectrums,
                match_tolerance,
                key,
            )

        # column suffix: first run = "_b10", later runs = "_b10_2", "_b10_3", ...
        suffix = "" if i == 1 else f"_{i}"

        b10_matches_pos_copy.loc[iter_idx, f"library_best_match_b10{suffix}"] = best_lib
        b10_matches_pos_copy.loc[iter_idx, f"library_best_match_num_matchpeak_b10{suffix}"] = best_npeaks
        b10_matches_pos_copy.loc[iter_idx, f"library_best_match_query_b10{suffix}"] = best_query
        b10_matches_pos_copy.loc[iter_idx, f"library_best_match_score_value_b10{suffix}"] = best_score

In [45]:
b10_matches_pos_curated = select_final_library_match_b10(b10_matches_pos_copy)

In [ ]:
# b10_exp_pos1 = load_experiment(b10dda_pos1)
# b10_ms2_pos_list1 = extract_ms2_with_precursors(b10_exp_pos1)
# b10_exp_pos2 = load_experiment(b10dda_pos2)
# b10_ms2_pos_list2 = extract_ms2_with_precursors(b10_exp_pos2)

# matches_pos = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv')
# b10_matches_pos_copy = matches_pos.copy()

# b10_candidates_df, b10_mz_col, b10_rt_col, b10_inchikey_col = load_candidates(
#     r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv',
#     mz_col="Average Mz",
#     rt_col="Average Rt(min)",
#     inchikey_col="InChiKey_origin",
# )

# b10_matches = match_candidates_to_ms2(
#     b10_ms2_pos_list1,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )

# b10_candidates_df, b10_mz_col, b10_rt_col, b10_inchikey_col = load_candidates(
#     r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv',
#     mz_col="Average Mz",
#     rt_col="Average Rt(min)",
#     inchikey_col="InChiKey_origin",
# )

# b10_matches2 = match_candidates_to_ms2(
#     b10_ms2_pos_list2,
#     b10_candidates_df,
#     b10_mz_col,
#     b10_rt_col,
#     b10_inchikey_col,
#     ppm_tolerance=10.0,
#     rt_tolerance_min=0.5,
# )


# peak_norm_tol = 0.01  # whatever you used before
# b10_query_spectra_by_inchikey = build_query_spectra_from_matches(
#     b10_matches, baseline=peak_norm_tol, normalize_and_filter=normalize_and_filter
# )
# b10_query_spectra_by_inchikey2 = build_query_spectra_from_matches(
#     b10_matches2, baseline=peak_norm_tol, normalize_and_filter=normalize_and_filter
# )

# for iter_idx, row in b10_matches_pos_copy.iterrows():
#     key = row['InChiKey_origin']

#     # --------------- QUERY SPECTRA (experimental) ---------------
#     # use spectra built from the AutoMSMS mzML
#     query_spectrums = b10_query_spectra_by_inchikey.get(key, [])
#     query_spectrums2 = b10_query_spectra_by_inchikey2.get(key,[])

#     if len(query_spectrums) == 0:
#         # print(f'No query spectrum found for compound {key}')
#         continue

#     if len(query_spectrums2) ==0:
#         continue

#     # --------------- REFERENCE SPECTRA (libraries) ---------------
#     reference_spectrums_library = []
#     sources = [
#         (mbank_data_pos,          reference_spectrums_library, 'mbank'),
#         (mona_data_pos,           reference_spectrums_library, 'mona'),
#         (mona_obtrap_data_pos,    reference_spectrums_library, 'mona_obtrap'),
#         (inhouse_data_pos,        reference_spectrums_library, 'inhouse1'),
#         (inhouse_data_pos2,       reference_spectrums_library, 'inhouse2'),
#         # (insilico_data_neg,    reference_spectrums_insilico, 'insilico'),  # example
#     ]

#     for data, library, prefix in sources:
#         spectra_dict = get_spectrum_by_key(data, key)  # your existing function

#         # spectra_dict: e.g. { "10eV": [spectrum1, spectrum2], "20eV": [...], ... }
#         for ce, spectra in spectra_dict.items():
#             for sspectrum in spectra:
#                 mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
#                 library.append(
#                     Spectrum(
#                         mz=mz,
#                         intensities=intensities,
#                         metadata={
#                             'inchikey': key,
#                             'peak_comments': f"{prefix}_{ce}"
#                         }
#                     )
#                 )

#     # --------------- MATCHING (your existing function) ---------------
#     best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
#         reference_spectrums_library,
#         query_spectrums,
#         match_tolerance,
#         key
#     )

#     # --------------- SAVE RESULTS ---------------
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_b10'] = best_spectrum_library
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_num_matchpeak_b10'] = best_num_matchpeak_library
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_query_b10'] = best_match_query_library
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_score_value_b10'] = best_score_library
    
#     best_score_library2, best_num_matchpeak_library2, best_spectrum_library2, best_match_query_library2 = find_best_match(
#         reference_spectrums_library,
#         query_spectrums2,
#         match_tolerance,
#         key
#     )
    
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_b10_2'] = best_spectrum_library2
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_num_matchpeak_b10_2'] = best_num_matchpeak_library2
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_query_b10_2'] = best_match_query_library2
#     b10_matches_pos_copy.loc[iter_idx, 'library_best_match_score_value_b10_2'] = best_score_library2

In [46]:
print(b10_matches_pos_copy['library_best_match_score_value_b10'].describe())
print(b10_matches_pos_copy['library_best_match_score_value_b10_5'].describe())

count    648.000000
mean       0.623064
std        0.277325
min        0.034494
25%        0.495045
50%        0.658321
75%        0.839894
max        0.998833
Name: library_best_match_score_value_b10, dtype: float64
count    437.000000
mean       0.384425
std        0.281153
min        0.006797
25%        0.103348
50%        0.377295
75%        0.603451
max        0.953654
Name: library_best_match_score_value_b10_5, dtype: float64


In [47]:
#combine the findings from targetedmsms, automsms, and targetedmsms from first 200
b10_matches_pos_curated['RTdiff'] = abs(b10_matches_pos_curated['Average Rt(min)']-b10_matches_pos_curated['RT_pos'])
b10_matches_pos_level1 = b10_matches_pos_curated[(b10_matches_pos_curated['library_best_match_score_value_b10_final']>=0.7)&(b10_matches_pos_curated['library_best_match_num_matchpeak_b10_final']>=3) & (b10_matches_pos_curated['RTdiff']<=0.5)]
print(b10_matches_pos_level1.shape)
b10_matches_pos_level2 = b10_matches_pos_curated[(b10_matches_pos_curated['library_best_match_score_value_b10_final']>=0.7)&(b10_matches_pos_curated['library_best_match_num_matchpeak_b10_final']>=3) & (b10_matches_pos_curated['RTdiff'].isna())]

#get unique DTXSID:
print(len(b10_matches_pos_level1['DTXSID_x'].unique()))
print(len(b10_matches_pos_level2['DTXSID_x'].unique()))

#combine the findings from targetedmsms, automsms, and targetedmsms from first 200
b10_matches_neg_curated['RTdiff'] = abs(b10_matches_neg_curated['Average Rt(min)']-b10_matches_neg_curated['RT_pos'])
b10_matches_neg_level1 = b10_matches_neg_curated[(b10_matches_neg_curated['library_best_match_score_value_b10_final']>=0.7)&(b10_matches_neg_curated['library_best_match_num_matchpeak_b10_final']>=3)&(b10_matches_neg_curated['RTdiff']<=0.5)]
print(b10_matches_neg_level1.shape)
b10_matches_neg_level2 = b10_matches_neg_curated[(b10_matches_neg_curated['library_best_match_score_value_b10_final']>=0.7)&(b10_matches_neg_curated['library_best_match_num_matchpeak_b10_final']>=3)&(b10_matches_neg_curated['RTdiff'].isna())]

#get unique DTXSID:
print(len(b10_matches_neg_level1['DTXSID_x'].unique()))
print(len(b10_matches_neg_level2['DTXSID_x'].unique()))

(36, 69)
10
28
(7, 85)
2
30


In [48]:
#remove rows with duplicated DTXSID and keep the first apperances
b10_pos_match_level1_unique = b10_matches_pos_level1.drop_duplicates(
    subset=["DTXSID_x"],
    keep="first"
).copy()
b10_neg_match_level1_unique = b10_matches_neg_level1.drop_duplicates(
    subset=["DTXSID_x"],
    keep="first"
).copy()

b10_pos_match_level1_unique['polarity'] = 'pos'
b10_neg_match_level1_unique['polarity'] = 'neg'
b10_pos_match_level1_unique['spectrum_acquiredby'] = 'dda'
b10_neg_match_level1_unique['spectrum_acquiredby'] = 'dda'

print(tabulate.tabulate(b10_pos_match_level1_unique, headers='keys', tablefmt='psql'))
print(tabulate.tabulate(b10_neg_match_level1_unique, headers='keys', tablefmt='psql'))

+-------+--------------+-------------------+--------------+------------------+---------------+----------------------------------------------+---------------------------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+-----------------------------+-----------------------------+-------------------+----------------+---------------------+-------------+---------------+----------+----------+------------------+------------------------+------------------+-------------------------------+------------------------+-------------------------+--------------------------+----------------------------------------+------------------------------------+--------------------------------------+----------------------------+------------------------------------------+------------------------------------+----------------------------------------+----------------------------+------------------------------------------+---------------

In [ ]:
#rowbind all level1 identifications
all_pos_level1_unique = all_pos_level1.drop_duplicates(
    subset=["DTXSID_x"],
    keep="first"
).copy()
all_neg_level1_unique = all_neg_level1.drop_duplicates(
    subset=["DTXSID_x"],
    keep="first"
).copy()
combined_level1_pos_neg = pd.concat([b10_pos_match_level1_unique,b10_neg_match_level1_unique,all_pos_level1_unique,all_neg_level1_unique])
# print(tabulate.tabulate(combined_level1_pos_neg, headers='keys', tablefmt='psql'))
print(len(combined_level1_pos_neg['DTXSID_x'].unique()))
combined_level1_pos_neg.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\combined_level1_pos_neg_byDec09.csv')

21


In [210]:
#get analysis for level1 compound,
#step 1: get unique DTXSID and inchikey from both pos and neg
#step 2: get the bloodpaper count, production volume, get the commercial use and function from the CDR data table by search againd CARSN
#step 3: get the detection frequency of each compound for level1 compound and draw a heatmap.
#step 4: get the detection frequency of each compound categories for level 2 compound and draw a heatmap

#import toxcast table for CARSN number
toxcast_table = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\suspect_chem_list_libraries\Total_Dsstox_data_valid.csv')
CDRdata = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\suspect_chem_list_libraries\2020 CDR Public CSV Data_0\2020 CDR Consumer and Commercial Use Information.csv')

#get CARSN from toxcast_table by search inchikey
toxcast_table2 = toxcast_table[['DTXSID','PREFERRED_NAME','CASRN','INCHIKEY']]
CDRdata2 = CDRdata[['CHEMICAL ID','CHEMICAL ID TYPE', 'CONS / COMM FUNCTION CATEGORY']]
CDRdata2['CASRN'] = CDRdata2['CHEMICAL ID']
tox_CDR= pd.merge(toxcast_table2,CDRdata2, on = 'CASRN', how = 'left')
combined_level1_pos_neg2 = combined_level1_pos_neg.iloc[:,0:12]
combined_level1_pos_neg2['DTXSID'] = combined_level1_pos_neg2['DTXSID_x']
#

level1_combined = pd.merge(combined_level1_pos_neg2,tox_CDR, on ='DTXSID', how = 'left')

level1_combined_unique = level1_combined .drop_duplicates(
    subset=["DTXSID"],
    keep="first"
).copy()

In [263]:
print(tabulate.tabulate(level1_combined_unique, headers= 'keys', tablefmt='psql'))
print(level1_combined_unique.shape)

+-----+--------------+-------------------+--------------+------------------+---------------+---------------------------------------------------------------------+---------------------------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+---------------+---------------------------------------+------------+-----------------------------+---------------+--------------------+---------------------------------------+
|     |   Unnamed: 0 |   Average Rt(min) |   Average Mz | feature_id       | DTXSID_x      | SMILES_STD                                                          | PREFERRED_NAME_x                      | MOLECULAR_FORMULA_original   | Pred. Ionization source   |   BloodPaperCount | 2019 PV   | InChiKey_origin             | DTXSID        | PREFERRED_NAME                        | CASRN      | INCHIKEY                    | CHEMICAL ID   | CHEMICAL ID TYPE   | CONS / COMM FUNCTION CATEGORY         |


In [ ]:
#plot the chemicals with RDkit
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from IPython.display import display

def draw_structure(SMILESlst, namelst):
    for a, b in zip(SMILESlst, namelst):
        m = Chem.MolFromSmiles(a)
        # tmp = AllChem.Compute2DCoords(m)
        # img = Draw.MolsToGridImage(m, molsPerRow=2, subImageSize = (150,150), legends = str(b))
        img = Draw.MolToImage(m, molPerRow=1)
        print(f'{str(b)}')
        display(img)
        savepath = os.path.join('D:/UCSF_postdoc_topic/REVEAL_first_400/Poolmsms_from_first400batch/results_level1_molecules/', f'{b}.png')
        Draw.MolToFile(m,savepath)

draw_structure(level1_combined_unique['SMILES_STD'].to_list(), level1_combined_unique['PREFERRED_NAME_x'].to_list())


In [1]:
##combine level 2 compounds and make summary
#processing candidates at level2

##do spectrum matching for new files
pos_newfilepath = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\acquisition_method\splitting_pool_samples\pos_level12_MS2_list.csv'
neg_newfilepath = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\acquisition_method\splitting_pool_samples\neg_level12_MS2_list.csv'

import pandas as pd
pos_level12_Dec15 = pd.read_csv(pos_newfilepath)
neg_level12_Dec15 = pd.read_csv(neg_newfilepath)

#merge with exact mass matches
posmatches_path = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak_removed redundant feature_compound matches for ms2_withpriority_20251209.csv'
negmatches_path = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak_removed redundant feature_compound matches for ms2_withpriority_20251209.csv'
pos_matches2= pd.read_csv(posmatches_path)
neg_matches2= pd.read_csv(negmatches_path)

# Filter rows where priority_confirmation is '1' or '2'
pos_matches2_level12 = pos_matches2[(pos_matches2['priority_confirmation']<=2)&(pos_matches2['priority_confirmation']>0)]
neg_matches2_level12 = neg_matches2[(neg_matches2['priority_confirmation']<=2)&(neg_matches2['priority_confirmation']>0)]

print("Filtered positive matches shape:", pos_matches2_level12.shape)

# Create feature_id by rounding float columns and combining as string
pos_level12_Dec15['feature_id'] = (
    pos_level12_Dec15['Average Mz'].astype(str) + '_' +
    pos_level12_Dec15['Average Rt(min)'].astype(str)
)
neg_level12_Dec15['feature_id'] = (
    neg_level12_Dec15['Average Mz'].astype(str) + '_' +
    neg_level12_Dec15['Average Rt(min)'].astype(str)
)

# Merge based on feature_id
pos_matches2_level12 = pd.merge(pos_matches2_level12, pos_level12_Dec15, on='feature_id', how='left')
neg_matches2_level12 = pd.merge(neg_matches2_level12, neg_level12_Dec15, on='feature_id', how='left')

C:\Users\yangj\AppData\Local\Temp\ipykernel_21064\1888574843.py:15: DtypeWarning: Columns (409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,433,434,440,442,445,448,451,454,457,458,459,460,466,470,474,475,481,482,491,496,497,500,501,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,571,572,573,574,575,577,578,579,580,581,582,583,584,585,587,605,606,607,608,609,610,613,616,617,618,619,620,621,630,631,632,636,637,638) have mixed types. Specify dtype option on import or set low_memory=False.
  pos_matches2= pd.read_csv(posmatches_path)
C:\Users\yangj\AppData\Local\Temp\ipykernel_21064\1888574843.py:16: DtypeWarning: Columns (410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,434,435,441,443,446,449,45

Filtered positive matches shape: (6046, 647)
